# 🩺 Modelo 3 — Mamografías: SANA vs REQUIERE EVALUACIÓN (RSNA), con más sanas y menos sobreajuste

**Qué cambia respecto al modelo 2** (la arquitectura, el tamaño de entrada y el preprocesamiento son los MISMOS, así la página no cambia):

| | Modelo 2 | Modelo 3 (este notebook) |
|---|---|---|
| Imágenes sanas | ~7.000 fijas (6 por cada cáncer) | **Hasta 30.000**, y en cada época se usa un grupo distinto |
| Imágenes por época | ~5.800 | **12.000** (todas las de cáncer + sanas que rotan) |
| Aumento de datos | Zoom leve, brillo, contraste | **Moderado:** + giro de ±10° y gamma (simula distintos mamógrafos) |
| Regularización | Dropout 0,4 | **Dropout 0,5 + weight decay** |
| Prueba final | 1.143 imágenes | **Las MISMAS 1.143** (mismos pacientes) → comparación justa |
| Comparación | — | **v2 vs v3** en el mismo test, por imagen y **por mama** (CC + MLO juntas) |
| Si se corta Colab | Empezar de nuevo | **Retoma** el fine-tuning desde la última época guardada en Drive |

**Antes de empezar (una sola vez):**
1. `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`.
2. El Secreto `KAGGLE_API_TOKEN` (ícono 🔑) con "Acceso del notebook" activado, igual que en el modelo 2.
3. El modelo 2 debe estar en tu Drive, en `modelo_mamografia_v2/modelo_mamografia_v2.keras` (lo dejó ahí el notebook 2). Si no está, la sección 12 te pide subir `modelo_mamografia_v2_web.zip`.
4. `Entorno de ejecución → Ejecutar todas` y acepta el permiso de **Google Drive**.

⏱️ Tiempo aproximado en T4: **2,5 a 3,5 horas** (descarga ~10 min, preprocesar ~32.000 imágenes ~40–60 min, entrenamiento ~90–120 min).
💾 Usa ~5 GB de tu Google Drive para guardar las imágenes ya preprocesadas (así, si Colab se corta, no hay que repetir ese paso). Puedes desactivarlo con `GUARDAR_CACHE_EN_DRIVE = False`.

> ⚠️ Herramienta académica / de apoyo. **No es un diagnóstico médico.**

## 1. Instalación e imports

In [ ]:
!pip install -q -U kagglehub

import os, re, io, json, glob, time, zipfile, shutil, warnings
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                             average_precision_score, precision_recall_curve)
import kagglehub

warnings.filterwarnings('ignore')
print("TensorFlow:", tf.__version__, "| kagglehub:", kagglehub.__version__)
GPUS = tf.config.list_physical_devices('GPU')
print("GPU:", GPUS if GPUS else "⚠️ NO HAY GPU → Entorno de ejecución > Cambiar tipo > T4 GPU")

## 2. Configuración (lo único que normalmente tocarías)

In [ ]:
SEED = 42

# --- Datos ---
DATASET_PNG = 'theoviel/rsna-breast-cancer-512-pngs'   # RSNA convertido a PNG 512x512
COMPETENCIA = 'rsna-breast-cancer-detection'           # de aquí salen las etiquetas (train.csv)

# --- Para reproducir EXACTAMENTE la división del modelo 2 (no cambiar) ---
NEGATIVOS_POR_POSITIVO_V2 = 6
INCLUIR_BIOPSIAS_NEGATIVAS = True
FRACCION_TEST = 0.15              # por paciente
FRACCION_VAL  = 0.15              # por paciente

# --- Modelo 3: más imágenes sanas ---
IMAGENES_POR_EPOCA = 12000        # todas las de cáncer + biopsias negativas + sanas que ROTAN cada época
MAX_SANAS_EN_POOL  = 30000        # sanas disponibles para rotar (más = más tiempo de preprocesado y más Drive)

# --- Aumento de datos MODERADO (solo en entrenamiento) ---
ROTACION_MAX_GRADOS = 10          # giro aleatorio entre -10° y +10°
ZOOM_MIN            = 0.85        # recorte aleatorio del 85% al 100% del área (zoom + traslación leves)
BRILLO_MAX          = 20.0        # ± en una escala de 0-255
CONTRASTE           = (0.8, 1.2)
GAMMA               = (0.85, 1.15)  # simula mamógrafos con distinta "curva" de grises

# --- Imagen (el MISMO preprocesamiento que usa la página) ---
BASE = 512
IMG_H, IMG_W = 512, 320

# --- Entrenamiento ---
BATCH_SIZE   = 16
EPOCHS_HEAD  = 2                  # fase 1: solo la cabeza
EPOCHS_FINE  = 14                 # fase 2: fine-tuning (EarlyStopping corta antes si no mejora)
LR_HEAD      = 1e-3
LR_FINE      = 3e-5
WEIGHT_DECAY = 1e-4               # regularización extra en el fine-tuning
DROPOUT      = 0.5
PESOS_BASE   = 'imagenet'
USAR_MIXED_PRECISION = True       # ~2x más rápido en T4

# --- Semáforo (se eligen en VALIDACIÓN, igual que en el modelo 2) ---
SENSIBILIDAD_OBJETIVO = 0.85
ESPECIFICIDAD_ROJO    = 0.95

# --- Guardado y reanudación ---
GUARDAR_EN_DRIVE       = True     # modelo, avances y resultados sobreviven si se cae la sesión
GUARDAR_CACHE_EN_DRIVE = True     # ~5 GB: imágenes ya preprocesadas (evita repetir 40-60 min)
REANUDAR               = True     # si hay un fine-tuning a medias en Drive, sigue desde ahí
RUTA_DATOS_LOCAL = None           # (solo para pruebas fuera de Colab; déjalo en None)
RUTA_MODELO_V2   = None           # None = se busca en Drive (modelo_mamografia_v2/modelo_mamografia_v2.keras)
PROBAR_MIS_IMAGENES = False       # True para subir tus propias mamografías en la sección 14

OUT_DIR = '/content/modelo_mamografia_v3'
DRIVE_V2 = None
if GUARDAR_EN_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUT_DIR = '/content/drive/MyDrive/modelo_mamografia_v3'
        DRIVE_V2 = '/content/drive/MyDrive/modelo_mamografia_v2/modelo_mamografia_v2.keras'
    except ImportError:
        print("No estás en Colab: se guarda en carpeta local.")
os.makedirs(OUT_DIR, exist_ok=True)
CARPETA_RAPIDA = '/content' if os.path.isdir('/content') else OUT_DIR   # disco local (rápido) para el caché

tf.keras.utils.set_random_seed(SEED)
if USAR_MIXED_PRECISION and GPUS:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
print("Política de precisión:", tf.keras.mixed_precision.global_policy().name)
print(f"Entrada: {IMG_H}x{IMG_W} | Imágenes por época: {IMAGENES_POR_EPOCA:,} | Salida en: {OUT_DIR}")

## 3. Llave de Kaggle (desde los Secretos de Colab)
La llave **no se escribe en el notebook**: se lee del Secreto `KAGGLE_API_TOKEN`.

In [ ]:
def configurar_kaggle():
    token = os.environ.get('KAGGLE_API_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('KAGGLE_API_TOKEN')
        except Exception as e:
            raise RuntimeError(
                "No encontré el Secreto 'KAGGLE_API_TOKEN'. Agrégalo en el ícono 🔑 de la barra "
                "izquierda y activa 'Acceso del notebook'. Detalle: " + repr(e))
    if not token or not token.strip():
        raise RuntimeError("El Secreto 'KAGGLE_API_TOKEN' está vacío.")
    os.environ['KAGGLE_API_TOKEN'] = token.strip()
    print("✅ Llave de Kaggle cargada (no se muestra por seguridad).")
    try:
        print("   Conectado como:", kagglehub.whoami())
    except Exception as e:
        print("   ⚠️ No pude verificar la cuenta (puede ser normal):", repr(e)[:120])

if RUTA_DATOS_LOCAL is None:
    configurar_kaggle()

## 4. Descarga de las imágenes y las etiquetas

In [ ]:
if RUTA_DATOS_LOCAL:
    CARPETA_PNG = RUTA_DATOS_LOCAL
else:
    print("Descargando imágenes (puede tardar ~10 min la primera vez)...")
    CARPETA_PNG = kagglehub.dataset_download(DATASET_PNG)
print("Carpeta de imágenes:", CARPETA_PNG)

PNGS = sorted(glob.glob(os.path.join(CARPETA_PNG, '**', '*.png'), recursive=True))
print(f"PNG encontrados: {len(PNGS):,}")
assert PNGS, "No encontré imágenes PNG en la descarga."

csvs = [c for c in glob.glob(os.path.join(CARPETA_PNG, '**', '*.csv'), recursive=True)
        if os.path.basename(c).lower() == 'train.csv']
if csvs:
    RUTA_CSV = csvs[0]
else:
    RUTA_CSV = kagglehub.competition_download(COMPETENCIA, path='train.csv')
    if os.path.isdir(RUTA_CSV):
        RUTA_CSV = os.path.join(RUTA_CSV, 'train.csv')
print("Etiquetas:", RUTA_CSV)

meta = pd.read_csv(RUTA_CSV)
meta.columns = meta.columns.str.strip().str.lower()
print(f"Filas en train.csv: {len(meta):,}")
print(meta['cancer'].value_counts().rename({0: 'sin cáncer', 1: 'cáncer'}))

In [ ]:
# Vincular cada PNG con su fila de train.csv usando (patient_id, image_id)
PATRON = re.compile(r'(\d+)[_/](\d+)\.png$', re.IGNORECASE)
filas = []
for p in PNGS:
    m = PATRON.search(p.replace('\\', '/'))
    if m:
        filas.append((int(m.group(1)), int(m.group(2)), p))
archivos = pd.DataFrame(filas, columns=['patient_id', 'image_id', 'ruta'])

df = meta.merge(archivos, on=['patient_id', 'image_id'], how='inner')
print(f"Imágenes vinculadas a una etiqueta: {len(df):,}")
assert len(df) > 0, "No pude vincular imágenes con etiquetas."
if 'biopsy' not in df: df['biopsy'] = 0
if 'laterality' not in df: df['laterality'] = 'X'

## 5. Preprocesamiento (el MISMO archivo que usa la página)
Escala de grises → `BASE x BASE` → invertir si el fondo es claro → **mama a la izquierda** → borrar lo que no es mama → **recorte de la mama** → `IMG_H x IMG_W`.
Es una copia exacta de `prevencion/services/preprocesamiento_v2.py` de la página: el modelo 3 se entrena con lo mismo que verá en la web.

In [ ]:
PREPRO_WEB = r"""

# Preprocesamiento del modelo 2 (idéntico en el notebook y en la web).
# Solo NumPy + Pillow.
import numpy as np
from PIL import Image

BASE, IMG_H, IMG_W = 512, 512, 320


def a_gris(imagen):
    # Imagen PIL -> arreglo float32 en 0-1 (acepta 8 y 16 bits, color o gris)
    if imagen.mode in ('I;16', 'I;16L', 'I;16B', 'I'):
        a = np.asarray(imagen, dtype=np.float32)
        return np.clip(a / (65535.0 if a.max() > 255 else 255.0), 0, 1)
    return np.asarray(imagen.convert('L'), dtype=np.float32) / 255.0


def redimensionar(a, ancho, alto):
    return np.asarray(Image.fromarray(a.astype(np.float32), mode='F').resize((ancho, alto), Image.BILINEAR))


def tramo_mas_largo(mascara_1d):
    # Inicio y fin (exclusivo) del tramo de True más largo
    mejor, inicio, mejor_ini = 0, None, 0
    for i, v in enumerate(list(mascara_1d) + [False]):
        if v and inicio is None:
            inicio = i
        elif not v and inicio is not None:
            if i - inicio > mejor:
                mejor, mejor_ini = i - inicio, inicio
            inicio = None
    return (mejor_ini, mejor_ini + mejor) if mejor else None


def limpiar_fuera_de_mama(a, umbral=0.10, hueco=12, inicio_max=0.30, largo_min=0.10):
    # La mama (ya a la izquierda) parte cerca del borde del tórax. En cada fila se conserva
    # solo el TRAMO de tejido más largo que empieza cerca de ese borde. Las etiquetas
    # ('RCC'), marcadores, reglas y barras son tramos cortos o separados: se borran.
    # Huecos oscuros de menos de `hueco` píxeles (zonas de grasa) se consideran tejido.
    alto, ancho = a.shape
    limpia = np.zeros_like(a)
    brillante = a > umbral
    for y in range(alto):
        d = np.diff(np.concatenate(([0], brillante[y].astype(np.int8), [0])))
        inicios, fines = list(np.flatnonzero(d == 1)), list(np.flatnonzero(d == -1))
        if not inicios:
            continue
        tramos = [[inicios[0], fines[0]]]
        for i0, f0 in zip(inicios[1:], fines[1:]):
            if i0 - tramos[-1][1] < hueco:
                tramos[-1][1] = f0            # une tramos separados por un hueco pequeño
            else:
                tramos.append([i0, f0])
        validos = [t for t in tramos if t[0] <= inicio_max * ancho and t[1] - t[0] >= largo_min * ancho]
        if validos:
            x0, x1 = max(validos, key=lambda t: t[1] - t[0])
            limpia[y, x0:x1] = a[y, x0:x1]
    return limpia


def caja_mama(a, umbral=0.08, fraccion=0.02, margen=0.02, minimo=0.25):
    # Caja (y0, y1, x0, x1) de la mama: el bloque brillante más grande.
    # Los textos y marcadores quedan fuera porque son tramos cortos.
    alto, ancho = a.shape
    mascara = a > umbral
    cols = tramo_mas_largo(mascara.mean(axis=0) > fraccion)
    if cols is None:
        return 0, alto, 0, ancho
    x0, x1 = cols
    filas = tramo_mas_largo(mascara[:, x0:x1].mean(axis=1) > fraccion)
    if filas is None:
        return 0, alto, 0, ancho
    y0, y1 = filas
    mx, my = int(margen * ancho), int(margen * alto)
    x0, x1 = max(0, x0 - mx), min(ancho, x1 + mx)
    y0, y1 = max(0, y0 - my), min(alto, y1 + my)
    if (x1 - x0) < minimo * ancho or (y1 - y0) < minimo * alto:
        return 0, alto, 0, ancho      # recorte dudoso: se usa la imagen completa
    return y0, y1, x0, x1


def preparar_imagen(imagen):
    # Imagen PIL -> uint8 (IMG_H, IMG_W) lista para el modelo
    a = redimensionar(a_gris(imagen), BASE, BASE)

    # Imagen invertida (fondo blanco): se invierte solo si así tiene claramente
    # más fondo negro. (El notebook miraba solo el borde, pero muchas películas
    # escaneadas tienen un marco blanco y no están invertidas. En RSNA, con
    # fondo negro, ambos criterios dan lo mismo: el modelo no cambia.)
    if (a < 0.15).mean() < (a > 0.85).mean():
        a = 1.0 - a

    mitad = BASE // 2
    if a[:, mitad:].sum() > a[:, :mitad].sum():   # mama siempre a la izquierda
        a = a[:, ::-1]

    limpia = limpiar_fuera_de_mama(a)
    if (limpia > 0.10).mean() < 0.05:   # limpieza dudosa (casi nada quedó): se usa la imagen tal cual
        limpia = a
    y0, y1, x0, x1 = caja_mama(limpia)
    a = redimensionar(np.ascontiguousarray(limpia[y0:y1, x0:x1]), IMG_W, IMG_H)
    return np.clip(np.round(a * 255.0), 0, 255).astype(np.uint8)


def a_entrada(img_u8):
    # uint8 (H, W) -> float32 (1, H, W, 3) en 0-255 (EfficientNet normaliza internamente)
    return np.repeat(img_u8[None, :, :, None].astype(np.float32), 3, axis=-1)
"""

with open(os.path.join(OUT_DIR, 'preprocesamiento_v2.py'), 'w', encoding='utf-8') as f:
    f.write(PREPRO_WEB.lstrip('\n'))
exec(PREPRO_WEB)   # define preparar_imagen(), a_entrada(), etc.
assert (BASE, IMG_H, IMG_W) == (512, 512, 320), "El preprocesamiento de la web usa 512 / 512x320"
print("✅ preprocesamiento_v2.py (idéntico al de la página) guardado en", OUT_DIR)

## 6. Mismos pacientes de validación y prueba que el modelo 2 + más sanas para entrenar
1. Se **repite exactamente** la selección y la división del modelo 2 (misma semilla) para recuperar sus pacientes de **validación** y **prueba**. Así la comparación v2 vs v3 es justa.
2. Para **entrenar**, se usan todas las imágenes de los demás pacientes: todas las de cáncer, todas las biopsias negativas y hasta `MAX_SANAS_EN_POOL` sanas.
3. Ninguna imagen de un paciente de validación o prueba entra al entrenamiento.

In [ ]:
# --- 6.1 Reproducir la selección y la división del modelo 2 ---
positivos = df[df.cancer == 1]
negativos = df[df.cancer == 0]
neg_biopsia = negativos[negativos.biopsy == 1] if INCLUIR_BIOPSIAS_NEGATIVAS else negativos.iloc[:0]
neg_resto = negativos.drop(neg_biopsia.index)
n_extra = max(0, NEGATIVOS_POR_POSITIVO_V2 * len(positivos) - len(neg_biopsia))
n_extra = min(n_extra, len(neg_resto))
seleccion_v2 = pd.concat([positivos, neg_biopsia, neg_resto.sample(n_extra, random_state=SEED)])
seleccion_v2 = seleccion_v2.sample(frac=1, random_state=SEED).reset_index(drop=True)

pacientes = seleccion_v2.groupby('patient_id')['cancer'].max()
def dividir(ids, etiquetas, fraccion, semilla):
    n = max(2, round(1 / fraccion))
    sgkf = StratifiedGroupKFold(n_splits=n, shuffle=True, random_state=semilla)
    a, b = next(sgkf.split(ids, etiquetas, groups=ids))
    return ids[a], ids[b]
ids = pacientes.index.values
ids_resto, ids_test = dividir(ids, pacientes.values, FRACCION_TEST, SEED)
ids_train_v2, ids_val = dividir(ids_resto, pacientes.loc[ids_resto].values,
                                FRACCION_VAL / (1 - FRACCION_TEST), SEED + 1)

df_val  = seleccion_v2[seleccion_v2.patient_id.isin(ids_val)].reset_index(drop=True)
df_test = seleccion_v2[seleccion_v2.patient_id.isin(ids_test)].reset_index(drop=True)
n_train_v2 = int(seleccion_v2.patient_id.isin(ids_train_v2).sum())

print(f"Modelo 2 → train {n_train_v2:,} | val {len(df_val):,} | test {len(df_test):,} imágenes")
if (len(df_val), len(df_test), int(df_test.cancer.sum())) == (1162, 1143, 183):
    print("✅ Validación y prueba son EXACTAMENTE las del modelo 2 (1.162 y 1.143 imágenes, 183 cánceres en prueba)")
else:
    print("⚠️ La división no coincide con la del modelo 2 (¿cambió el dataset?). La comparación v2 vs v3 "
          "sigue siendo en el mismo test, pero el modelo 2 pudo haber visto algunos de estos pacientes.")

# --- 6.2 Pool de entrenamiento del modelo 3: todos los demás pacientes ---
fuera = set(ids_val) | set(ids_test)
pool = df[~df.patient_id.isin(fuera)]
pool_pos  = pool[pool.cancer == 1]
pool_bio  = pool[(pool.cancer == 0) & (pool.biopsy == 1)] if INCLUIR_BIOPSIAS_NEGATIVAS else pool.iloc[:0]
pool_sana = pool[(pool.cancer == 0) & ~pool.index.isin(pool_bio.index)]
pool_sana = pool_sana.sample(min(MAX_SANAS_EN_POOL, len(pool_sana)), random_state=SEED)
df_train = (pd.concat([pool_pos, pool_bio, pool_sana])
              .sort_values(['patient_id', 'image_id']).reset_index(drop=True))
df_train['fija'] = (df_train.cancer == 1) | (df_train.biopsy == 1)   # siempre en cada época

assert not set(df_train.patient_id) & fuera, "Fuga: un paciente de val/test quedó en entrenamiento"
assert not set(df_val.patient_id) & set(df_test.patient_id), "Fuga val/test"
for nombre, d in [('train', df_train), ('val', df_val), ('test', df_test)]:
    assert d.cancer.nunique() == 2, f"{nombre} no tiene ambas clases"

n_fijas = int(df_train.fija.sum())
SANAS_POR_EPOCA = int(min(max(0, IMAGENES_POR_EPOCA - n_fijas), (~df_train.fija).sum()))
print(f"\nModelo 3 → pool de entrenamiento: {len(df_train):,} imágenes "
      f"({len(pool_pos):,} con cáncer, {len(pool_bio):,} biopsias negativas, {len(pool_sana):,} sanas)")
print(f"Cada época: {n_fijas:,} fijas + {SANAS_POR_EPOCA:,} sanas distintas = {n_fijas + SANAS_POR_EPOCA:,} imágenes")
print(f"En {EPOCHS_HEAD + EPOCHS_FINE} épocas el modelo puede ver hasta "
      f"~{min(len(pool_sana), SANAS_POR_EPOCA * (EPOCHS_HEAD + EPOCHS_FINE)):,} sanas distintas "
      f"(el modelo 2 veía siempre las mismas {n_train_v2 - int(seleccion_v2[seleccion_v2.patient_id.isin(ids_train_v2)].cancer.sum()):,})")

## 7. Preprocesar una sola vez (con caché en Drive)
Cada imagen se recorta y achica **una vez**. Las de entrenamiento se guardan en un archivo en disco (no caben en la memoria de Colab) y, si `GUARDAR_CACHE_EN_DRIVE = True`, se copian a Drive: si la sesión se corta, la próxima vez se cargan en ~5 minutos en vez de repetir este paso.

In [ ]:
import sys, multiprocessing

# Los procesos de trabajo importan el preprocesamiento desde un archivo (así funcionan en
# paralelo de verdad y sin tocar TensorFlow)
CARPETA_COD = os.path.join(CARPETA_RAPIDA, 'codigo_v3')
os.makedirs(CARPETA_COD, exist_ok=True)
with open(os.path.join(CARPETA_COD, 'preprocesamiento_v2.py'), 'w', encoding='utf-8') as f:
    f.write(PREPRO_WEB.lstrip('\n'))
with open(os.path.join(CARPETA_COD, 'trabajador_v3.py'), 'w', encoding='utf-8') as f:
    f.write("from PIL import Image\nfrom preprocesamiento_v2 import preparar_imagen\n\n"
            "def una(ruta):\n    with Image.open(ruta) as im:\n        return preparar_imagen(im)\n")
if CARPETA_COD not in sys.path:
    sys.path.insert(0, CARPETA_COD)
import trabajador_v3

def preprocesar(rutas, destino):
    """Llena `destino` (array o memmap) con las imágenes preprocesadas, en paralelo."""
    t0 = time.time()
    with ProcessPoolExecutor(max_workers=os.cpu_count() or 2,
                             mp_context=multiprocessing.get_context('spawn')) as ex:
        for i, img in enumerate(ex.map(trabajador_v3.una, rutas, chunksize=32)):
            destino[i] = img
            if (i + 1) % 2000 == 0:
                print(f"  {i + 1:,}/{len(rutas):,} ({(time.time() - t0) / 60:.1f} min)")
    return destino

def cargar_o_preprocesar(d, nombre):
    """Caché en disco local (+ copia en Drive). Devuelve un memmap de solo lectura."""
    huella = f"{len(d)}_{int(d.image_id.sum()) % 10**9}"
    local = os.path.join(CARPETA_RAPIDA, f'cache_{nombre}_{huella}.npy')
    en_drive = os.path.join(OUT_DIR, os.path.basename(local))
    if not os.path.exists(local) and os.path.exists(en_drive):
        print(f"Copiando caché desde Drive ({os.path.getsize(en_drive) / 1e9:.1f} GB)...")
        shutil.copy(en_drive, local)
    if os.path.exists(local):
        X = np.load(local, mmap_mode='r')
        if X.shape == (len(d), IMG_H, IMG_W):
            print(f"✅ {nombre}: caché reutilizado ({len(d):,} imágenes)")
            return X
    print(f"Preprocesando {nombre} ({len(d):,} imágenes)...")
    X = np.lib.format.open_memmap(local + '.tmp', mode='w+', dtype=np.uint8, shape=(len(d), IMG_H, IMG_W))
    preprocesar(d.ruta.tolist(), X); X.flush(); del X
    os.replace(local + '.tmp', local)
    if GUARDAR_CACHE_EN_DRIVE and local != en_drive:
        print("Copiando caché a Drive...")
        shutil.copy(local, en_drive)
    return np.load(local, mmap_mode='r')

X_train = cargar_o_preprocesar(df_train, 'train_v3')
print("Preprocesando val y test...")
X_val  = preprocesar(df_val.ruta.tolist(),  np.zeros((len(df_val),  IMG_H, IMG_W), np.uint8))
X_test = preprocesar(df_test.ruta.tolist(), np.zeros((len(df_test), IMG_H, IMG_W), np.uint8))
y_train, y_val, y_test = (d.cancer.values.astype('float32') for d in (df_train, df_val, df_test))
print(f"✅ Listo: train {X_train.shape} (en disco), val {X_val.shape}, test {X_test.shape}")

## 8. Pipeline de entrenamiento: sanas que rotan + aumento de datos moderado
- En cada época entran **todas** las imágenes fijas (cáncer y biopsias negativas) y un grupo **distinto** de sanas.
- Como hay ~9 sanas por cada cáncer, los cánceres pesan más en el entrenamiento (`PESOS_CLASE`), para que el modelo no se acostumbre a decir "sana".
- Aumento moderado: zoom y traslación leves, giro de ±10°, brillo, contraste y gamma. **No** se voltea (la mama siempre queda a la izquierda).

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
idx_fijas = np.flatnonzero(df_train.fija.values)
idx_sanas = np.flatnonzero(~df_train.fija.values)
POR_EPOCA = len(idx_fijas) + SANAS_POR_EPOCA
PASOS_POR_EPOCA = POR_EPOCA // BATCH_SIZE

n_pos = int(y_train[idx_fijas].sum())
PESOS_CLASE = {0: POR_EPOCA / (2 * (POR_EPOCA - n_pos)), 1: POR_EPOCA / (2 * n_pos)}
print(f"Por época: {POR_EPOCA:,} imágenes ({n_pos:,} con cáncer, {100 * n_pos / POR_EPOCA:.1f}%) | "
      f"{PASOS_POR_EPOCA} pasos de {BATCH_SIZE}")
print("Pesos por clase:", {k: round(v, 3) for k, v in PESOS_CLASE.items()})

def generador_train():
    rng = np.random.default_rng(SEED)
    while True:                                   # una vuelta = una época con sanas distintas
        idx = np.concatenate([idx_fijas, rng.choice(idx_sanas, SANAS_POR_EPOCA, replace=False)])
        rng.shuffle(idx)
        for i in idx:
            yield X_train[i], y_train[i]

def rotar(img, grados):
    # Giro alrededor del centro; lo que queda fuera se rellena con negro (fondo de la mamografía)
    a = grados * np.pi / 180.0
    c, s = tf.cos(a), tf.sin(a)
    cx, cy = (IMG_W - 1) / 2.0, (IMG_H - 1) / 2.0
    t = tf.stack([c, -s, cx - c * cx + s * cy, s, c, cy - s * cx - c * cy, 0.0, 0.0])[None]
    return tf.raw_ops.ImageProjectiveTransformV3(
        images=img[None], transforms=t, output_shape=[IMG_H, IMG_W],
        fill_value=0.0, interpolation='BILINEAR', fill_mode='CONSTANT')[0]

def aumentar(img):
    img = tf.cast(img[..., None], tf.float32)
    escala = tf.random.uniform([], ZOOM_MIN, 1.0)                     # zoom + traslación leves
    y0 = tf.random.uniform([], 0.0, 1.0 - escala)
    x0 = tf.random.uniform([], 0.0, 1.0 - escala)
    caja = tf.reshape(tf.stack([y0, x0, y0 + escala, x0 + escala]), (1, 4))
    img = tf.image.crop_and_resize(img[None], caja, [0], (IMG_H, IMG_W))[0]
    img = rotar(img, tf.random.uniform([], -ROTACION_MAX_GRADOS, ROTACION_MAX_GRADOS))
    img = tf.image.random_brightness(img, BRILLO_MAX)
    img = tf.image.random_contrast(img, *CONTRASTE)
    img = tf.clip_by_value(img, 0.0, 255.0)
    gamma = tf.random.uniform([], *GAMMA)
    img = 255.0 * tf.pow(img / 255.0, gamma)
    return tf.image.grayscale_to_rgb(img)

def a_entrada_modelo(img):
    return tf.image.grayscale_to_rgb(tf.cast(img[..., None], tf.float32))

w_clase = tf.constant([PESOS_CLASE[0], PESOS_CLASE[1]], tf.float32)
train_ds = (tf.data.Dataset.from_generator(
                generador_train,
                output_signature=(tf.TensorSpec((IMG_H, IMG_W), tf.uint8), tf.TensorSpec((), tf.float32)))
            .map(lambda x, l: (aumentar(x), l, tf.gather(w_clase, tf.cast(l, tf.int32))),
                 num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE).prefetch(AUTOTUNE))

def crear_ds_eval(X, y):
    return (tf.data.Dataset.from_tensor_slices((X, y))
            .map(lambda x, l: (a_entrada_modelo(x), l), num_parallel_calls=AUTOTUNE)
            .batch(BATCH_SIZE).prefetch(AUTOTUNE))
val_ds, test_ds = crear_ds_eval(X_val, y_val), crear_ds_eval(X_test, y_test)

In [ ]:
# Vista del aumento de datos: la misma imagen 6 veces (debe seguir viéndose como una mamografía normal)
ejemplo = X_train[idx_fijas[0]]
fig, ax = plt.subplots(1, 7, figsize=(16, 4))
ax[0].imshow(ejemplo, cmap='gray', vmin=0, vmax=255); ax[0].set_title('original', fontsize=9)
for i in range(1, 7):
    ax[i].imshow(aumentar(ejemplo).numpy()[..., 0], cmap='gray', vmin=0, vmax=255)
    ax[i].set_title(f'aumentada {i}', fontsize=9)
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()

## 9. Modelo (EfficientNetB0, igual que el modelo 2 → la página y el Grad-CAM no cambian)

In [ ]:
def construir_modelo(pesos=PESOS_BASE):
    base = tf.keras.applications.EfficientNetB0(include_top=False, weights=pesos,
                                                input_shape=(IMG_H, IMG_W, 3))
    base.trainable = False
    entradas = tf.keras.Input(shape=(IMG_H, IMG_W, 3), name='imagen')
    x = base(entradas, training=False)          # BatchNorm nunca actualiza estadísticas
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(DROPOUT, name='dropout')(x)
    salida = layers.Dense(1, activation='sigmoid', dtype='float32', name='prob_maligno')(x)
    return tf.keras.Model(entradas, salida, name='clasificador_mamografia_v3'), base

def metricas():
    return [tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.AUC(curve='PR', name='pr_auc'),
            tf.keras.metrics.Recall(name='sensibilidad'),
            tf.keras.metrics.Precision(name='precision')]

def descongelar(base):
    nombres = [l.name for l in base.layers]
    corte = next((i for i, n in enumerate(nombres) if n.startswith('block4a')), len(nombres) - 100)
    for i, capa in enumerate(base.layers):
        capa.trainable = (i >= corte) and not isinstance(capa, layers.BatchNormalization)

CKPT   = os.path.join(OUT_DIR, 'mejor_modelo_v3.keras')    # el mejor según AUC de validación
ULTIMO = os.path.join(OUT_DIR, 'ultimo_v3.keras')          # el de la última época (para reanudar)
ESTADO = os.path.join(OUT_DIR, 'estado_v3.json')

## 10. Fase 1 — entrenar solo la cabeza (se salta si hay un fine-tuning para reanudar)

In [ ]:
estado = None
if REANUDAR and os.path.exists(ESTADO) and os.path.exists(ULTIMO):
    with open(ESTADO, encoding='utf-8') as f:
        estado = json.load(f)
    print(f"↻ Reanudando: el fine-tuning llegó a la época {estado['epoca'] + 1} "
          f"(mejor AUC de validación hasta ahora: {estado['mejor_auc']:.4f})")

model, base = construir_modelo()
if estado is None:
    model.compile(optimizer=tf.keras.optimizers.Adam(LR_HEAD), loss='binary_crossentropy', metrics=metricas())
    cb1 = [tf.keras.callbacks.ModelCheckpoint(CKPT, monitor='val_auc', mode='max', save_best_only=True, verbose=1)]
    h1 = model.fit(train_ds, steps_per_epoch=PASOS_POR_EPOCA, validation_data=val_ds,
                   epochs=EPOCHS_HEAD, callbacks=cb1)
    MEJOR_AUC_F1 = max(h1.history['val_auc'])
    print(f"Mejor val AUC fase 1: {MEJOR_AUC_F1:.4f}")
else:
    h1 = None

## 11. Fase 2 — fine-tuning (bloques 4–7, BatchNorm congelado, AdamW)
Al final de **cada época** se guarda el avance en Drive. Si Colab se corta, vuelve a `Ejecutar todas`: el caché y el avance se recuperan y el entrenamiento sigue desde la última época.

In [ ]:
class GuardarAvance(tf.keras.callbacks.Callback):
    """Guarda el modelo de la última época + en qué época va, para poder reanudar."""
    def __init__(self, mejor_inicial):
        super().__init__()
        self.mejor = mejor_inicial
    def on_epoch_end(self, epoch, logs=None):
        self.mejor = max(self.mejor, float((logs or {}).get('val_auc', 0.0)))
        self.model.save(ULTIMO)
        with open(ESTADO, 'w', encoding='utf-8') as f:
            json.dump({'epoca': int(epoch), 'mejor_auc': self.mejor}, f)

if estado is None:
    ep_inicio, mejor_previo = EPOCHS_HEAD, MEJOR_AUC_F1
    base.trainable = True
    descongelar(base)
else:
    ep_inicio, mejor_previo = estado['epoca'] + 1, estado['mejor_auc']
    model = tf.keras.models.load_model(ULTIMO)
    base = next(l for l in model.layers if isinstance(l, tf.keras.Model))
    base.trainable = True
    descongelar(base)
print(f"Capas entrenables en la base: {sum(c.trainable for c in base.layers)} / {len(base.layers)}")

model.compile(optimizer=tf.keras.optimizers.AdamW(LR_FINE, weight_decay=WEIGHT_DECAY),
              loss='binary_crossentropy', metrics=metricas())
cb2 = [
    tf.keras.callbacks.ModelCheckpoint(CKPT, monitor='val_auc', mode='max', save_best_only=True,
                                       initial_value_threshold=mejor_previo, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=False),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=2,
                                         min_lr=1e-7, verbose=1),
    GuardarAvance(mejor_previo),
]
h2 = model.fit(train_ds, steps_per_epoch=PASOS_POR_EPOCA, validation_data=val_ds,
               epochs=EPOCHS_HEAD + EPOCHS_FINE, initial_epoch=ep_inicio, callbacks=cb2)

In [ ]:
# Curvas de aprendizaje (si se reanudó, solo muestran la parte de esta sesión)
hist = {k: (h1.history.get(k, []) if h1 else []) + h2.history[k] for k in h2.history}
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for a, m, t in zip(ax, ['loss', 'auc', 'pr_auc'], ['Pérdida', 'AUC', 'PR-AUC']):
    a.plot(hist[m], label='Entrenamiento'); a.plot(hist['val_' + m], label='Validación')
    a.set_title(t); a.set_xlabel('Época'); a.legend()
plt.tight_layout(); plt.show()
brecha = hist['auc'][-1] - hist['val_auc'][-1]
print(f"Brecha AUC entrenamiento - validación al final: {brecha:.3f} "
      f"(el modelo 2 tenía ~0,24: 0,88 vs 0,64). Más chica = menos sobreajuste.")

model = tf.keras.models.load_model(CKPT)   # el mejor según val AUC
print("✅ Cargado el mejor modelo:", CKPT)

## 12. Umbrales del semáforo (en validación) y evaluación final en TEST
Igual que en el modelo 2: **Verde** bajo `UMBRAL` (≥85% de los cánceres de validación quedan sobre él), **Rojo** desde `UMBRAL_ROJO` (solo ~5% de las sanas llega). El test solo se usa aquí.

In [ ]:
def umbrales(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    thr = np.clip(thr, 0, 1)
    u = float(thr[np.argmax(tpr >= SENSIBILIDAD_OBJETIVO)])
    ok = np.where(fpr <= 1 - ESPECIFICIDAD_ROJO)[0]
    ur = float(thr[ok[-1]]) if len(ok) else 0.9
    return u, max(ur, u + 1e-6)

def nivel(p, u, ur):
    return 'Verde' if p < u else ('Amarillo' if p < ur else 'Rojo')

def auc_ic(y, p, n=1000):
    rng, vals = np.random.default_rng(SEED), []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2:
            vals.append(roc_auc_score(y[i], p[i]))
    return np.percentile(vals, [2.5, 97.5])

def reporte(y, p, u, ur, nombre, mostrar=True):
    auc = roc_auc_score(y, p); lo, hi = auc_ic(y, p)
    tn, fp, fn, tp = confusion_matrix(y, (p >= u).astype(int), labels=[0, 1]).ravel()
    r = {'AUC': auc, 'AUC_IC95': [float(lo), float(hi)], 'PR_AUC': average_precision_score(y, p),
         'sensibilidad': tp / (tp + fn), 'especificidad': tn / (tn + fp),
         'VPP': tp / (tp + fp) if tp + fp else 0.0, 'VPN': tn / (tn + fn) if tn + fn else 0.0,
         'n': int(len(y)), 'prevalencia': float(np.mean(y))}
    sem = pd.crosstab(pd.Series(y).map({0: 'SIN CÁNCER', 1: 'CÁNCER'}).rename('real'),
                      pd.Series([nivel(v, u, ur) for v in p]).rename('semáforo')
                      ).reindex(columns=['Verde', 'Amarillo', 'Rojo'], fill_value=0)
    if mostrar:
        print(f"\n=== {nombre} (umbrales {u:.3f} / {ur:.3f}) ===")
        print(f"AUC: {auc:.4f}  (IC95%: {lo:.3f}–{hi:.3f}) | PR-AUC: {r['PR_AUC']:.3f} (azar = {r['prevalencia']:.3f})")
        for k in ['sensibilidad', 'especificidad', 'VPP', 'VPN']:
            print(f"{k:>14}: {r[k]:.3f}")
        print("\nSemáforo por clase real:\n", sem)
    return r, sem, np.array([[tn, fp], [fn, tp]])

p_val  = model.predict(val_ds,  verbose=0).ravel().astype('float64')
p_test = model.predict(test_ds, verbose=0).ravel().astype('float64')
UMBRAL, UMBRAL_ROJO = umbrales(y_val, p_val)
print(f"UMBRAL (Verde→Amarillo): {UMBRAL:.4f} | UMBRAL_ROJO (Amarillo→Rojo): {UMBRAL_ROJO:.4f}")
r_val, _, _              = reporte(y_val,  p_val,  UMBRAL, UMBRAL_ROJO, 'MODELO 3 · VALIDACIÓN')
r_test, sem_test, cm_test = reporte(y_test, p_test, UMBRAL, UMBRAL_ROJO, 'MODELO 3 · TEST (resultado final)')

## 13. ¿Es mejor que el modelo 2? (mismo test, mismos pacientes)
Se carga el modelo 2 y se evalúa con **exactamente las mismas imágenes**. También se mide **por mama**: el índice de una mama es el promedio de sus vistas (CC y MLO), como lo leería un radiólogo.
La diferencia de AUC se acompaña de un intervalo (bootstrap pareado): si todo el intervalo está sobre 0, la mejora es real y no suerte.

In [ ]:
def buscar_modelo_v2():
    for ruta in [RUTA_MODELO_V2, DRIVE_V2, '/content/modelo_mamografia_v2.keras']:
        if ruta and os.path.exists(ruta):
            return ruta
    try:
        from google.colab import files
        print("No encontré el modelo 2 en Drive. Sube modelo_mamografia_v2_web.zip "
              "(está en la carpeta modelo_cnn_v2 del proyecto):")
        for nombre, contenido in files.upload().items():
            with zipfile.ZipFile(io.BytesIO(contenido)) as z:
                z.extract('modelo_mamografia_v2.keras', '/content')
            return '/content/modelo_mamografia_v2.keras'
    except ImportError:
        pass
    raise FileNotFoundError("No encontré el modelo 2: define RUTA_MODELO_V2 en la configuración.")

tf.keras.mixed_precision.set_global_policy('float32')
modelo_v2 = tf.keras.models.load_model(buscar_modelo_v2())
pv2_val  = modelo_v2.predict(val_ds,  verbose=0).ravel().astype('float64')
pv2_test = modelo_v2.predict(test_ds, verbose=0).ravel().astype('float64')
U2, UR2 = umbrales(y_val, pv2_val)
r2_test, sem2_test, _ = reporte(y_test, pv2_test, U2, UR2, 'MODELO 2 · TEST', mostrar=False)

def por_mama(d, p):
    g = d.assign(p=p).groupby(['patient_id', 'laterality'])
    return g.cancer.max().values.astype(float), g.p.mean().values

ym, pm3 = por_mama(df_test, p_test)
_,  pm2 = por_mama(df_test, pv2_test)

def dif_auc_pareada(y, pa, pb, n=1000):
    rng, d = np.random.default_rng(SEED), []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2:
            d.append(roc_auc_score(y[i], pa[i]) - roc_auc_score(y[i], pb[i]))
    return float(np.mean(d)), np.percentile(d, [2.5, 97.5])

dif_img, ic_img = dif_auc_pareada(y_test, p_test, pv2_test)
dif_mama, ic_mama = dif_auc_pareada(ym, pm3, pm2)

def fila(r, sem):
    return {'AUC': f"{r['AUC']:.3f} ({r['AUC_IC95'][0]:.2f}–{r['AUC_IC95'][1]:.2f})",
            'Sensibilidad': f"{r['sensibilidad']:.0%}", 'Especificidad': f"{r['especificidad']:.0%}",
            'VPN': f"{r['VPN']:.0%}", 'VPP': f"{r['VPP']:.0%}",
            'Cánceres en Verde': int(sem.loc['CÁNCER', 'Verde']) if 'CÁNCER' in sem.index else 0,
            'Sanas en Rojo': int(sem.loc['SIN CÁNCER', 'Rojo']) if 'SIN CÁNCER' in sem.index else 0}

comparacion = pd.DataFrame({'Modelo 2': fila(r2_test, sem2_test), 'Modelo 3': fila(r_test, sem_test)}).T
comparacion['AUC por mama'] = [f"{roc_auc_score(ym, pm2):.3f}", f"{roc_auc_score(ym, pm3):.3f}"]
print(f"TEST: {len(y_test):,} imágenes · {len(ym):,} mamas · {int(y_test.sum())} imágenes con cáncer\n")
print(comparacion.to_string())
print(f"\nDiferencia de AUC (v3 - v2) por imagen: {dif_img:+.3f}  (IC95%: {ic_img[0]:+.3f} a {ic_img[1]:+.3f})")
print(f"Diferencia de AUC (v3 - v2) por mama:   {dif_mama:+.3f}  (IC95%: {ic_mama[0]:+.3f} a {ic_mama[1]:+.3f})")

if ic_img[0] > 0:
    VEREDICTO = "✅ El modelo 3 es MEJOR que el modelo 2 (la mejora no es suerte). Conviene usarlo en la página."
elif dif_img > 0.01:
    VEREDICTO = ("🟡 El modelo 3 sale algo mejor, pero la diferencia podría ser suerte. Mira también "
                 "especificidad, 'Sanas en Rojo' y el AUC por mama antes de decidir.")
elif dif_img >= -0.01:
    VEREDICTO = ("🟡 Los dos modelos son prácticamente iguales en AUC (diferencia menor a 0,01). Solo conviene "
                 "cambiar si el modelo 3 mejora claramente la especificidad sin perder sensibilidad.")
else:
    VEREDICTO = "❌ El modelo 3 NO supera al modelo 2: nos quedamos con el modelo 2."
print("\n" + VEREDICTO)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for y, p, n in [(y_test, pv2_test, 'Modelo 2'), (y_test, p_test, 'Modelo 3')]:
    f, t, _ = roc_curve(y, p); ax[0].plot(f, t, label=f"{n} (AUC={roc_auc_score(y, p):.3f})")
ax[0].plot([0, 1], [0, 1], 'k--', alpha=.4)
ax[0].set_xlabel('1 - Especificidad'); ax[0].set_ylabel('Sensibilidad'); ax[0].set_title('Curva ROC (TEST, por imagen)'); ax[0].legend()
for y, p, n in [(ym, pm2, 'Modelo 2'), (ym, pm3, 'Modelo 3')]:
    f, t, _ = roc_curve(y, p); ax[1].plot(f, t, label=f"{n} (AUC={roc_auc_score(y, p):.3f})")
ax[1].plot([0, 1], [0, 1], 'k--', alpha=.4)
ax[1].set_xlabel('1 - Especificidad'); ax[1].set_ylabel('Sensibilidad'); ax[1].set_title('Curva ROC (TEST, por mama)'); ax[1].legend()
ax[2].imshow(cm_test, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax[2].text(j, i, cm_test[i, j], ha='center', va='center', fontsize=16,
                   color='white' if cm_test[i, j] > cm_test.max() / 2 else 'black')
ax[2].set_xticks([0, 1], ['SIN CÁNCER', 'CÁNCER']); ax[2].set_yticks([0, 1], ['SIN CÁNCER', 'CÁNCER'])
ax[2].set_xlabel('Predicción'); ax[2].set_ylabel('Real'); ax[2].set_title('Matriz de confusión modelo 3 (TEST)')
plt.tight_layout(); plt.show()

## 14. Prueba de estabilidad, Grad-CAM y tus propias imágenes
**Estabilidad:** recortar solo 2% de los bordes no debería cambiar el resultado (modelo 2: 12% de las imágenes cambiaba de nivel).

In [ ]:
def predecir_imagen(imagen, m=None):
    x = a_entrada(preparar_imagen(imagen))
    return float((m or model)(x, training=False)[0, 0])

muestra_est = df_test.sample(min(200, len(df_test)), random_state=SEED)
difs, cambios = [], 0
for r in muestra_est['ruta']:
    with Image.open(r) as im:
        im = im.convert('L'); w, h = im.size
        p1 = predecir_imagen(im)
        p2 = predecir_imagen(im.crop((int(w * .02), int(h * .02), int(w * .98), int(h * .98))))
    difs.append(abs(p1 - p2) * 100); cambios += nivel(p1, UMBRAL, UMBRAL_ROJO) != nivel(p2, UMBRAL, UMBRAL_ROJO)
print(f"Recortar 2% de los bordes ({len(difs)} imágenes de test):")
print(f"  cambio medio {np.mean(difs):.1f} pts | máximo {np.max(difs):.1f} pts | "
      f"cambian de nivel {cambios} de {len(difs)} ({100 * cambios / len(difs):.0f}%)")
print("  (modelo 2: medio 4.9 pts, 12% cambiaba de nivel)")

In [ ]:
# Grad-CAM: con GAP + una neurona, el peso de cada canal es el peso de la capa final (igual que en la web)
base_m = next(l for l in model.layers if isinstance(l, tf.keras.Model))
pesos_cabeza = model.get_layer('prob_maligno').get_weights()[0][:, 0]

def mapa_calor(img_u8):
    act = base_m(a_entrada(img_u8), training=False)[0].numpy().astype('float32')
    calor = np.maximum((act * pesos_cabeza).sum(-1), 0)
    calor = calor / (calor.max() + 1e-8)
    return np.asarray(Image.fromarray(calor, mode='F').resize((IMG_W, IMG_H), Image.BILINEAR))

ejemplos = pd.concat([df_test[df_test.cancer == 1].sample(min(5, int(df_test.cancer.sum())), random_state=1),
                      df_test[df_test.cancer == 0].sample(5, random_state=1)])
fig, ax = plt.subplots(2, len(ejemplos), figsize=(2.2 * len(ejemplos), 8))
for i, (_, r) in enumerate(ejemplos.iterrows()):
    with Image.open(r['ruta']) as im:
        img = preparar_imagen(im)
    p = float(model(a_entrada(img), training=False)[0, 0])
    ax[0, i].imshow(img, cmap='gray'); ax[0, i].set_title('CÁNCER' if r.cancer else 'SIN CÁNCER', fontsize=9)
    ax[1, i].imshow(img, cmap='gray'); ax[1, i].imshow(mapa_calor(img), cmap='jet', alpha=0.4)
    ax[1, i].set_title(f"{nivel(p, UMBRAL, UMBRAL_ROJO)} ({p:.0%})", fontsize=9)
    ax[0, i].axis('off'); ax[1, i].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Tus propias mamografías (jpg/png): pon PROBAR_MIS_IMAGENES = True en la configuración
if PROBAR_MIS_IMAGENES:
    try:
        from google.colab import files
        for nombre, contenido in files.upload().items():
            with Image.open(io.BytesIO(contenido)) as im:
                p3, p2 = predecir_imagen(im), predecir_imagen(im, modelo_v2)
            print(f"{nombre} → modelo 3: {nivel(p3, UMBRAL, UMBRAL_ROJO)} ({p3:.0%}) | "
                  f"modelo 2: {nivel(p2, U2, UR2)} ({p2:.0%})")
    except ImportError:
        print("Esta celda solo funciona en Google Colab.")

## 15. Exportar para la web (modelo float32 + config + preprocesamiento)

In [ ]:
tf.keras.mixed_precision.set_global_policy('float32')
# Se construye con pesos='imagenet' a propósito: EfficientNet agrega su capa de escala solo así
# (con weights=None no existe y el modelo exportado procesaría distinto). Luego se ponen los entrenados.
modelo_web, _ = construir_modelo(pesos=PESOS_BASE)
modelo_web.set_weights(model.get_weights())
RUTA_WEB = os.path.join(OUT_DIR, 'modelo_mamografia_v3.keras')
modelo_web.save(RUTA_WEB)

x_prueba = a_entrada(X_test[0])
dif = abs(float(modelo_web(x_prueba)[0, 0]) - float(model(x_prueba)[0, 0]))
print(f"Diferencia float32 vs original: {dif:.5f}")
assert dif < 0.01, "El modelo exportado no coincide con el entrenado: NO lo uses y avísame."

def a_float(v):
    return v if isinstance(v, (list, int)) else float(v)

config = {
    'version': 'v3', 'dataset': f'RSNA Screening Mammography ({DATASET_PNG})',
    'pregunta': 'probabilidad de CÁNCER (sigmoid) - sana/benigna vs requiere evaluación',
    'base': BASE, 'img_h': IMG_H, 'img_w': IMG_W,
    'umbral': UMBRAL, 'umbral_rojo': UMBRAL_ROJO,
    'sensibilidad_objetivo': SENSIBILIDAD_OBJETIVO, 'especificidad_rojo': ESPECIFICIDAD_ROJO,
    'backbone': 'EfficientNetB0',
    'entrada': 'ver preprocesamiento_v2.py (el mismo de la web): gris -> BASExBASE -> invertir si fondo claro -> '
               'mama a la izquierda -> recorte de la mama -> IMG_HxIMG_W -> uint8 -> 3 canales float32 0-255',
    'entrenamiento': {'imagenes_por_epoca': int(POR_EPOCA), 'pool_entrenamiento': int(len(df_train)),
                      'sanas_en_pool': int(len(idx_sanas)), 'fijas_por_epoca': int(len(idx_fijas)),
                      'aumento': {'rotacion_grados': ROTACION_MAX_GRADOS, 'zoom_min': ZOOM_MIN,
                                  'brillo': BRILLO_MAX, 'contraste': list(CONTRASTE), 'gamma': list(GAMMA)},
                      'dropout': DROPOUT, 'weight_decay': WEIGHT_DECAY},
    'metricas_test': {k: a_float(v) for k, v in r_test.items()},
    'metricas_val':  {k: a_float(v) for k, v in r_val.items()},
    'metricas_test_por_mama': {'AUC_v3': float(roc_auc_score(ym, pm3)), 'AUC_v2': float(roc_auc_score(ym, pm2)),
                               'n_mamas': int(len(ym))},
    'comparacion_v2': {'metricas_test_v2': {k: a_float(v) for k, v in r2_test.items()},
                       'dif_auc_imagen': dif_img, 'dif_auc_imagen_IC95': [float(x) for x in ic_img],
                       'dif_auc_mama': dif_mama, 'dif_auc_mama_IC95': [float(x) for x in ic_mama],
                       'veredicto': VEREDICTO},
    'estabilidad_recorte_2pct': {'cambio_medio_pts': float(np.mean(difs)),
                                 'cambian_de_nivel_pct': float(100 * cambios / len(difs))},
    'n_imagenes': {'train_pool': int(len(df_train)), 'val': int(len(y_val)), 'test': int(len(y_test))},
    'tensorflow': tf.__version__,
}
with open(os.path.join(OUT_DIR, 'config.json'), 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

ZIP = os.path.join('/content' if os.path.isdir('/content') else OUT_DIR, 'modelo_mamografia_v3_web.zip')
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for n in ['modelo_mamografia_v3.keras', 'config.json', 'preprocesamiento_v2.py']:
        z.write(os.path.join(OUT_DIR, n), n)
print("📦 Paquete listo:", ZIP, f"({os.path.getsize(ZIP) / 1e6:.1f} MB)")
print("\n" + VEREDICTO)
try:
    from google.colab import files
    files.download(ZIP)
except ImportError:
    pass

## ✅ Listo
Se descargó `modelo_mamografia_v3_web.zip` (también queda en tu Drive, carpeta `modelo_mamografia_v3`) con el modelo, `config.json` (umbrales, métricas y la comparación con el modelo 2) y el preprocesamiento.

**Siguiente paso:** descarga también este notebook con sus resultados (`Archivo → Descargar → .ipynb`) y deja ambos archivos en la carpeta `modelo_cnn_v3` del proyecto. Con eso reviso los resultados y, **solo si el modelo 3 gana**, lo convierto a ONNX y lo cambio en la página.

**Cómo leer el resultado:** la tabla de la sección 13. Lo más importante es la **diferencia de AUC** con su intervalo, la **especificidad** (menos sanas en Amarillo) y que la **sensibilidad** siga cerca de 85%.

**Si Colab se cortó:** vuelve a `Entorno de ejecución → Ejecutar todas`. El caché de imágenes y el avance del fine-tuning se recuperan desde Drive.

**Al terminar:** puedes borrar de tu Drive los archivos `cache_train_v3_*.npy` y `ultimo_v3.keras` para recuperar espacio.